## Preprocessing for the Bayesian Network


Prima di procedere con l'apprendimento probabilistico, è necessario adattare il dataset alle specifiche delle Reti Bayesiane (BN). A differenza dei modelli di regressione (come Random Forest usato nel Modulo 1), le BN lavorano in modo ottimale con stati discreti e richiedono una struttura causale definita a priori (Expert Knowledge).

Operazioni Svolte:

Discretizzazione (Binning): Conversione delle variabili continue in categorie qualitative per ridurre la complessità computazionale e migliorare l'interpretabilità.

Età: Mappata in fasce demografiche (Giovane, Adulto, Senior).

Target: Il tempo di consegna (minuti) è stato convertito in una variabile binaria di stato (Puntuale vs Ritardo) usando una soglia critica (45 min).

Definizione Topologia (DAG): Costruzione del Grafo Diretto Aciclico che rappresenta le dipendenze condizionali. La struttura è stata progettata seguendo la logica causale del dominio logistico: le condizioni ambientali (Meteo) influenzano le condizioni operative (Traffico), che a loro volta, insieme ai fattori umani (Età), determinano il successo della consegna.

In [4]:
"""
==============================================================================
AMAZON DELIVERY RISK ANALYSIS - RETE BAYESIANA
Parte 1: Preprocessing, Discretizzazione e Definizione Struttura DAG
==============================================================================

Corso: Intelligenza Artificiale
Libreria: pgmpy (Probabilistic Graphical Models in Python)

TEORIA:
Le Reti Bayesiane sono modelli grafici probabilistici che rappresentano
le dipendenze condizionali tra variabili tramite un Grafo Aciclico Diretto (DAG).

Vantaggi:
- Rappresentazione compatta delle distribuzioni di probabilità congiunte
- Inferenza efficiente (ragionamento probabilistico)
- Interpretabilità: la struttura del grafo codifica relazioni causali
==============================================================================
"""

import pandas as pd
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork 
import warnings
warnings.filterwarnings('ignore')


# ==============================================================================
# SEZIONE 1: CARICAMENTO DATI
# ==============================================================================

def load_data(filepath: str) -> pd.DataFrame:
    """
    Carica il dataset Amazon Delivery ottimizzato.
    
    Args:
        filepath: Percorso del file CSV
        
    Returns:
        DataFrame con i dati grezzi
    """
    print("\n" + "="*70)
    print("  AMAZON DELIVERY RISK ANALYSIS - RETE BAYESIANA")
    print("  Parte 1: Preprocessing e Definizione Struttura")
    print("="*70)
    
    print(f"\n📂 STEP 1: Caricamento dati da: {filepath}")
    
    try:
        df = pd.read_csv(filepath)
        print(f"   ✅ Dataset caricato: {len(df)} righe, {len(df.columns)} colonne")
        print(f"\n   📋 Colonne disponibili:")
        for i, col in enumerate(df.columns, 1):
            print(f"      {i:2}. {col}")
        return df
    except FileNotFoundError:
        print(f"   ❌ File non trovato: {filepath}")
        raise


# ==============================================================================
# SEZIONE 2: DISCRETIZZAZIONE DELLE VARIABILI
# ==============================================================================

def discretize_age(age: float) -> str:
    """
    Discretizza l'età dell'autista in categorie.
    
    NOTA: Agent_Age nel dataset è già normalizzato (valori standardizzati).
    Usiamo i percentili per definire le categorie.
    
    Mapping (basato su valori normalizzati):
        < -0.5  -> 'Giovane'
        -0.5 a 0.5 -> 'Adulto'
        > 0.5   -> 'Senior'
    
    Args:
        age: Età normalizzata dell'autista
        
    Returns:
        Categoria di età come stringa
    """
    if pd.isna(age):
        return 'Adulto'
    elif age < -0.5:
        return 'Giovane'
    elif age <= 0.5:
        return 'Adulto'
    else:
        return 'Senior'


def discretize_delivery_time(minutes: float) -> str:
    """
    Discretizza il tempo di consegna in stato di puntualità.
    
    Mapping:
        <= 45 minuti  -> 'Puntuale'
        > 45 minuti   -> 'Ritardo'
    
    Args:
        minutes: Tempo di consegna in minuti
        
    Returns:
        Stato della consegna ('Puntuale' o 'Ritardo')
    """
    if pd.isna(minutes):
        return 'Puntuale'
    elif minutes <= 45:
        return 'Puntuale'
    else:
        return 'Ritardo'


def reverse_one_hot_weather(df: pd.DataFrame) -> pd.Series:
    """
    Ricostruisce la colonna categorica Weather_conditions
    dalle colonne One-Hot Encoded.
    
    Colonne One-Hot: Weather_Cloudy, Weather_Fog, Weather_Sandstorms,
                     Weather_Stormy, Weather_Sunny, Weather_Windy
    
    Args:
        df: DataFrame con colonne one-hot
        
    Returns:
        Series con valori categorici
    """
    weather_cols = ['Weather_Cloudy', 'Weather_Fog', 'Weather_Sandstorms',
                    'Weather_Stormy', 'Weather_Sunny', 'Weather_Windy']
    
    # Verifica quali colonne esistono
    existing_cols = [col for col in weather_cols if col in df.columns]
    
    if not existing_cols:
        return pd.Series(['Sunny'] * len(df), index=df.index)
    
    # Ricostruisci la categoria
    def get_weather(row):
        for col in existing_cols:
            if row[col] == 1:
                return col.replace('Weather_', '')
        return 'Sunny'  # Default
    
    return df[existing_cols].apply(get_weather, axis=1)


def reverse_one_hot_traffic(df: pd.DataFrame) -> pd.Series:
    """
    Ricostruisce la colonna categorica Road_traffic_density
    dalle colonne One-Hot Encoded.
    
    Colonne One-Hot: Traffic_High, Traffic_Jam, Traffic_Low, Traffic_Medium
    
    Args:
        df: DataFrame con colonne one-hot
        
    Returns:
        Series con valori categorici
    """
    traffic_cols = ['Traffic_High', 'Traffic_Jam', 'Traffic_Low', 'Traffic_Medium']
    
    # Verifica quali colonne esistono
    existing_cols = [col for col in traffic_cols if col in df.columns]
    
    if not existing_cols:
        return pd.Series(['Medium'] * len(df), index=df.index)
    
    # Ricostruisci la categoria
    def get_traffic(row):
        for col in existing_cols:
            if row[col] == 1:
                return col.replace('Traffic_', '')
        return 'Medium'  # Default
    
    return df[existing_cols].apply(get_traffic, axis=1)


def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applica tutte le trasformazioni di preprocessing e discretizzazione.
    
    Trasformazioni:
    1. Discretizzazione età autista (Agent_Age) -> 'Age_Category'
    2. Discretizzazione tempo consegna (Time_Minutes) -> 'Delivery_Status'
    3. Reverse One-Hot Encoding per Weather -> 'Weather_conditions'
    4. Reverse One-Hot Encoding per Traffic -> 'Road_traffic_density'
    
    Args:
        df: DataFrame originale
        
    Returns:
        DataFrame preprocessato con variabili discrete
    """
    print("\n🔄 STEP 2: Preprocessing e Discretizzazione")
    print("-"*70)
    
    # Crea una copia per non modificare l'originale
    df_processed = df.copy()
    
    # ──────────────────────────────────────────────────────────────
    # TASK 1a: Discretizzazione Età Autista (Agent_Age)
    # ──────────────────────────────────────────────────────────────
    print("\n   📊 Discretizzazione Età Autista (Agent_Age):")
    
    if 'Agent_Age' in df_processed.columns:
        # Mostra statistiche dei valori normalizzati
        print(f"      • Valori normalizzati - Min: {df_processed['Agent_Age'].min():.2f}, Max: {df_processed['Agent_Age'].max():.2f}")
        print(f"      • Media: {df_processed['Agent_Age'].mean():.2f}, Std: {df_processed['Agent_Age'].std():.2f}")
        
        # Applica la funzione di discretizzazione
        df_processed['Age_Category'] = df_processed['Agent_Age'].apply(discretize_age)
        
        # Stampa distribuzione
        age_dist = df_processed['Age_Category'].value_counts()
        total = len(df_processed)
        print(f"\n      Distribuzione:")
        print(f"      • Giovane (< -0.5):  {age_dist.get('Giovane', 0):>6} ({age_dist.get('Giovane', 0)/total*100:.1f}%)")
        print(f"      • Adulto (-0.5÷0.5): {age_dist.get('Adulto', 0):>6} ({age_dist.get('Adulto', 0)/total*100:.1f}%)")
        print(f"      • Senior (> 0.5):    {age_dist.get('Senior', 0):>6} ({age_dist.get('Senior', 0)/total*100:.1f}%)")
    else:
        print("      ⚠️ Colonna 'Agent_Age' non trovata")
        df_processed['Age_Category'] = 'Adulto'
    
    # ──────────────────────────────────────────────────────────────
    # TASK 1b: Discretizzazione Tempo Consegna -> Delivery_Status
    # ──────────────────────────────────────────────────────────────
    print("\n   📊 Discretizzazione Tempo Consegna (Time_Minutes):")
    
    if 'Time_Minutes' in df_processed.columns:
        print(f"      • Min: {df_processed['Time_Minutes'].min():.1f} min")
        print(f"      • Max: {df_processed['Time_Minutes'].max():.1f} min")
        print(f"      • Media: {df_processed['Time_Minutes'].mean():.1f} min")
        print(f"      • Mediana: {df_processed['Time_Minutes'].median():.1f} min")
        
        # Applica la funzione di discretizzazione
        df_processed['Delivery_Status'] = df_processed['Time_Minutes'].apply(discretize_delivery_time)
        
        # Stampa distribuzione
        status_dist = df_processed['Delivery_Status'].value_counts()
        total = len(df_processed)
        print(f"\n      Distribuzione:")
        print(f"      ✅ Puntuale (≤ 45 min): {status_dist.get('Puntuale', 0):>6} ({status_dist.get('Puntuale', 0)/total*100:.1f}%)")
        print(f"      ⚠️ Ritardo (> 45 min):  {status_dist.get('Ritardo', 0):>6} ({status_dist.get('Ritardo', 0)/total*100:.1f}%)")
    else:
        print("      ⚠️ Colonna 'Time_Minutes' non trovata")
        np.random.seed(42)
        df_processed['Delivery_Status'] = np.random.choice(
            ['Puntuale', 'Ritardo'], size=len(df_processed), p=[0.7, 0.3]
        )
    
    # ──────────────────────────────────────────────────────────────
    # TASK 1c: Reverse One-Hot Encoding per Weather_conditions
    # ──────────────────────────────────────────────────────────────
    print("\n   📊 Ricostruzione Weather_conditions (da One-Hot Encoding):")
    
    weather_cols = ['Weather_Cloudy', 'Weather_Fog', 'Weather_Sandstorms',
                    'Weather_Stormy', 'Weather_Sunny', 'Weather_Windy']
    existing_weather = [col for col in weather_cols if col in df_processed.columns]
    
    if existing_weather:
        print(f"      Colonne One-Hot trovate: {existing_weather}")
        
        # Reverse one-hot encoding
        df_processed['Weather_conditions'] = reverse_one_hot_weather(df_processed)
        
        # Stampa distribuzione
        weather_dist = df_processed['Weather_conditions'].value_counts()
        total = len(df_processed)
        print(f"\n      Distribuzione:")
        for condition, count in weather_dist.items():
            print(f"      • {condition:15s}: {count:>6} ({count/total*100:.1f}%)")
    else:
        print("      ⚠️ Colonne Weather One-Hot non trovate")
        df_processed['Weather_conditions'] = 'Sunny'
    
    # ──────────────────────────────────────────────────────────────
    # TASK 1d: Reverse One-Hot Encoding per Road_traffic_density
    # ──────────────────────────────────────────────────────────────
    print("\n   📊 Ricostruzione Road_traffic_density (da One-Hot Encoding):")
    
    traffic_cols = ['Traffic_High', 'Traffic_Jam', 'Traffic_Low', 'Traffic_Medium']
    existing_traffic = [col for col in traffic_cols if col in df_processed.columns]
    
    if existing_traffic:
        print(f"      Colonne One-Hot trovate: {existing_traffic}")
        
        # Reverse one-hot encoding
        df_processed['Road_traffic_density'] = reverse_one_hot_traffic(df_processed)
        
        # Stampa distribuzione
        traffic_dist = df_processed['Road_traffic_density'].value_counts()
        total = len(df_processed)
        print(f"\n      Distribuzione:")
        for density, count in traffic_dist.items():
            print(f"      • {density:15s}: {count:>6} ({count/total*100:.1f}%)")
    else:
        print("      ⚠️ Colonne Traffic One-Hot non trovate")
        df_processed['Road_traffic_density'] = 'Medium'
    
    return df_processed


# ==============================================================================
# SEZIONE 3: DEFINIZIONE STRUTTURA DAG (GRAFO ACICLICO DIRETTO)
# ==============================================================================

def define_bayesian_structure() -> DiscreteBayesianNetwork:
    """
    Definisce la struttura del DAG per la Rete Bayesiana.
    
    STRUTTURA CAUSALE:
    ──────────────────
    
    Weather_conditions ─────┬────────────────────┐
                            │                    │
                            ▼                    ▼
               Road_traffic_density ──────► Delivery_Status
                                                 ▲
                                                 │
                    Age_Category ────────────────┘
    
    
    ARCHI (EDGES):
    1. Weather_conditions -> Road_traffic_density
       (Il meteo influenza il traffico: pioggia/neve causano rallentamenti)
    
    2. Weather_conditions -> Delivery_Status
       (Il meteo influenza direttamente i tempi di consegna)
    
    3. Road_traffic_density -> Delivery_Status
       (Il traffico influenza i tempi di consegna)
    
    4. Age_Category -> Delivery_Status
       (L'esperienza dell'autista influenza la puntualità)
    
    Returns:
        Oggetto DiscreteBayesianNetwork con la struttura definita
    """
    print("\n🔗 STEP 3: Definizione Struttura DAG")
    print("-"*70)
    
    # ══════════════════════════════════════════════════════════════
    # DEFINIZIONE DEGLI ARCHI (RELAZIONI CAUSALI)
    # ══════════════════════════════════════════════════════════════
    
    edges = [
        # Il meteo influenza la densità del traffico
        ('Weather_conditions', 'Road_traffic_density'),
        
        # Il meteo influenza direttamente lo stato della consegna
        ('Weather_conditions', 'Delivery_Status'),
        
        # Il traffico influenza lo stato della consegna
        ('Road_traffic_density', 'Delivery_Status'),
        
        # L'età/esperienza dell'autista influenza la puntualità
        ('Age_Category', 'Delivery_Status')
    ]
    
    print("\n   📐 Archi del DAG (relazioni causali):")
    for parent, child in edges:
        print(f"      • {parent:25s} ──► {child}")
    
    # ══════════════════════════════════════════════════════════════
    # CREAZIONE DEL MODELLO BAYESIANO
    # ══════════════════════════════════════════════════════════════
    
    model = DiscreteBayesianNetwork(edges)
    
    print("\n   ✅ Modello DiscreteBayesianNetwork inizializzato")
    print(f"      • Nodi: {list(model.nodes())}")
    print(f"      • Archi: {len(list(model.edges()))}")
    
    # ══════════════════════════════════════════════════════════════
    # VERIFICA PROPRIETÀ DEL DAG
    # ══════════════════════════════════════════════════════════════
    
    print("\n   🔍 Proprietà del grafo:")
    
    # Identifica nodi radice (senza genitori)
    root_nodes = [node for node in model.nodes() if len(list(model.predecessors(node))) == 0]
    print(f"      • Nodi radice (senza genitori): {root_nodes}")
    
    # Identifica nodi foglia (senza figli)
    leaf_nodes = [node for node in model.nodes() if len(list(model.successors(node))) == 0]
    print(f"      • Nodi foglia (senza figli): {leaf_nodes}")
    
    # Mostra i genitori di ogni nodo
    print("\n   📋 Struttura delle dipendenze:")
    for node in model.nodes():
        parents = list(model.predecessors(node))
        if parents:
            print(f"      • P({node}) dipende da: {parents}")
        else:
            print(f"      • P({node}) è una distribuzione marginale (nodo radice)")
    
    return model


# ==============================================================================
# SEZIONE 4: PREPARAZIONE DATI PER IL TRAINING
# ==============================================================================

def prepare_training_data(df_processed: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara il DataFrame finale per il training della rete.
    
    Seleziona solo le colonne necessarie per la rete bayesiana
    e verifica che non ci siano valori mancanti.
    
    Args:
        df_processed: DataFrame preprocessato
        
    Returns:
        DataFrame pronto per il training
    """
    print("\n📦 STEP 4: Preparazione dati per il training")
    print("-"*70)
    
    # Colonne richieste per la rete bayesiana
    required_columns = [
        'Weather_conditions',
        'Road_traffic_density', 
        'Age_Category',
        'Delivery_Status'
    ]
    
    # Verifica che tutte le colonne esistano
    missing_cols = [col for col in required_columns if col not in df_processed.columns]
    if missing_cols:
        print(f"   ⚠️ Colonne mancanti: {missing_cols}")
        raise ValueError(f"Colonne mancanti nel DataFrame: {missing_cols}")
    
    # Seleziona solo le colonne necessarie
    df_training = df_processed[required_columns].copy()
    
    # Rimuovi eventuali righe con valori mancanti
    initial_rows = len(df_training)
    df_training = df_training.dropna()
    removed_rows = initial_rows - len(df_training)
    
    print(f"   ✅ Colonne selezionate: {required_columns}")
    print(f"   ✅ Righe totali: {len(df_training)}")
    if removed_rows > 0:
        print(f"   ⚠️ Righe rimosse (NaN): {removed_rows}")
    
    # Converti tutte le colonne a stringa (categorical)
    for col in df_training.columns:
        df_training[col] = df_training[col].astype(str)
    
    print("\n   📊 Riepilogo stati per ogni variabile:")
    for col in df_training.columns:
        unique_vals = df_training[col].unique()
        print(f"      • {col}: {list(unique_vals)}")
    
    return df_training


# ==============================================================================
# SEZIONE 5: VISUALIZZAZIONE STRUTTURA
# ==============================================================================

def visualize_dag_structure(model: DiscreteBayesianNetwork):
    """
    Stampa una rappresentazione testuale della struttura del DAG.
    
    Args:
        model: Modello DiscreteBayesianNetwork
    """
    print("\n" + "="*70)
    print("  STRUTTURA DEL DAG - RAPPRESENTAZIONE VISUALE")
    print("="*70)
    
    dag_visual = """
    
                    ┌─────────────────────┐
                    │  Weather_conditions │
                    └─────────┬───────────┘
                              │
              ┌───────────────┴───────────────┐
              │                               │
              ▼                               ▼
    ┌─────────────────────┐         ┌─────────────────────┐
    │Road_traffic_density │         │                     │
    └─────────┬───────────┘         │                     │
              │                     │                     │
              │   ┌─────────────────┘                     │
              │   │                                       │
              ▼   ▼                                       │
    ┌─────────────────────┐                               │
    │                     │◄──────────────────────────────┘
    │   Delivery_Status   │
    │                     │◄──────────────────────────────┐
    └─────────────────────┘                               │
                                                          │
                    ┌─────────────────────┐               │
                    │    Age_Category     │───────────────┘
                    └─────────────────────┘
    
    
    LEGENDA:
    ─────────
    • Weather_conditions: Condizioni meteorologiche (Sunny, Cloudy, Fog, etc.)
    • Road_traffic_density: Densità del traffico (Low, Medium, High, Jam)
    • Age_Category: Categoria età autista (Giovane, Adulto, Senior)
    • Delivery_Status: Stato consegna [TARGET] (Puntuale, Ritardo)
    
    """
    print(dag_visual)


# ==============================================================================
# MAIN - ESECUZIONE PARTE 1
# ==============================================================================

def main():
    """
    Funzione principale che orchestra la Parte 1:
    1. Caricamento dati
    2. Preprocessing e discretizzazione
    3. Definizione struttura DAG
    4. Preparazione dati per training
    """
    
    # ──────────────────────────────────────────────────────────────
    # STEP 1: Caricamento dati
    # ──────────────────────────────────────────────────────────────
    filepath = '../Csp/amazon_delivery_optimized.csv'
    
    try:
        df = load_data(filepath)
    except FileNotFoundError:
        # Prova percorso alternativo
        try:
            df = load_data('amazon_delivery_optimized.csv')
        except FileNotFoundError:
            print("\n❌ Impossibile trovare il file amazon_delivery_optimized.csv")
            print("   Assicurati che il file sia presente nella cartella corretta.")
            return None, None, None
    
    # ──────────────────────────────────────────────────────────────
    # STEP 2: Preprocessing e Discretizzazione
    # ──────────────────────────────────────────────────────────────
    df_processed = preprocess_data(df)
    
    # ──────────────────────────────────────────────────────────────
    # STEP 3: Definizione Struttura DAG
    # ──────────────────────────────────────────────────────────────
    model = define_bayesian_structure()
    
    # ──────────────────────────────────────────────────────────────
    # STEP 4: Preparazione dati per training
    # ──────────────────────────────────────────────────────────────
    df_training = prepare_training_data(df_processed)
    
    # ──────────────────────────────────────────────────────────────
    # Visualizzazione struttura
    # ──────────────────────────────────────────────────────────────
    visualize_dag_structure(model)
    
    # ──────────────────────────────────────────────────────────────
    # RIEPILOGO FINALE
    # ──────────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("  RIEPILOGO PARTE 1 - COMPLETATA")
    print("="*70)
    print(f"""
    ✅ OPERAZIONI COMPLETATE:
    
    1. Dataset caricato: {len(df)} righe
    
    2. Variabili discretizzate:
       • Age_Category: Giovane, Adulto, Senior (da Agent_Age normalizzato)
       • Delivery_Status: Puntuale, Ritardo (soglia 45 min)
       • Weather_conditions: ricostruito da One-Hot Encoding
       • Road_traffic_density: ricostruito da One-Hot Encoding
    
    3. Struttura DAG definita:
       • 4 nodi (variabili)
       • 4 archi (relazioni causali)
       • Variabile target: Delivery_Status
    
    4. Dati pronti per il training: {len(df_training)} righe
    
    
    📌 PROSSIMO STEP (Parte 2):
       - Apprendimento delle CPT (Conditional Probability Tables)
       - Maximum Likelihood Estimation (MLE)
       - Inferenza probabilistica
    """)
    print("="*70)
    
    return model, df_training, df_processed


# ==============================================================================
# ESECUZIONE
# ==============================================================================

if __name__ == "__main__":
    model, df_training, df_processed = main()
    
    # Salva il DataFrame processato per la Parte 2
    if df_training is not None:
        df_training.to_csv('bayesian_training_data.csv', index=False)
        print("\n💾 Dati di training salvati in: bayesian_training_data.csv")


  AMAZON DELIVERY RISK ANALYSIS - RETE BAYESIANA
  Parte 1: Preprocessing e Definizione Struttura

📂 STEP 1: Caricamento dati da: ../Csp/amazon_delivery_optimized.csv
   ✅ Dataset caricato: 39997 righe, 45 colonne

   📋 Colonne disponibili:
       1. Order_ID
       2. Agent_Age
       3. Agent_Rating
       4. Store_Latitude
       5. Store_Longitude
       6. Drop_Latitude
       7. Drop_Longitude
       8. Delivery_Time
       9. Weather_Cloudy
      10. Weather_Fog
      11. Weather_Sandstorms
      12. Weather_Stormy
      13. Weather_Sunny
      14. Weather_Windy
      15. Traffic_High
      16. Traffic_Jam
      17. Traffic_Low
      18. Traffic_Medium
      19. Vehicle_motorcycle
      20. Vehicle_scooter
      21. Vehicle_van
      22. Area_Metropolitian
      23. Area_Other
      24. Area_Semi-Urban
      25. Area_Urban
      26. Category_Apparel
      27. Category_Books
      28. Category_Clothing
      29. Category_Cosmetics
      30. Category_Electronics
      31. Categor

 ## Training (MLE) e InferenzaTitolo Attività: Apprendimento dei Parametri e Ragionamento Probabilistico
 
 Descrizione:Dopo aver definito la topologia della rete, questa fase trasforma il grafo qualitativo in un modello quantitativo funzionante.L'obiettivo è duplice: "insegnare" alla rete le probabilità basate sullo storico dati e utilizzare il modello addestrato per stimare rischi in condizioni di incertezza.Operazioni Svolte:Apprendimento Parametri (Training): Utilizzo dello stimatore di Massima Verosimiglianza (Maximum Likelihood Estimation - MLE). L'algoritmo calcola le Tabelle di Probabilità Condizionata (CPT) per ogni nodo misurando la frequenza relativa degli eventi nel dataset di training.Esempio: Calcolo di $P(\text{Ritardo} \mid \text{Meteo=Stormy}, \text{Traffic=Jam})$.Validazione Modello: Verifica della consistenza probabilistica (somma delle probabilità = 1) e della struttura (assenza di cicli).Inferenza (Querying): Implementazione di un motore di interrogazione basato sull'algoritmo di Eliminazione di Variabile (Variable Elimination). Questo permette di effettuare due tipi di ragionamento:Predittivo (Causale): Stimare la probabilità di un effetto (es. Ritardo) date le cause (es. Meteo, Età).Diagnostico (Evidenziale): Risalire alla probabilità delle cause (es. c'era traffico?) osservando l'effetto (es. il pacco è in ritardo).

In [ ]:
"""
=============================================================================
PARTE 2: TRAINING E INFERENZA DELLA RETE BAYESIANA
=============================================================================
Progetto: Sistema di Analisi Ritardi Consegne
Obiettivo: Apprendimento parametri (MLE) e Query probabilistiche
=============================================================================
"""

import pandas as pd
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# SEZIONE 1: CARICAMENTO DATI E COSTRUZIONE MODELLO
# ═══════════════════════════════════════════════════════════════════════════

def load_and_prepare_data(filepath='bayesian_training_data.csv'):
    """Carica e prepara i dati per il training"""
    
    print("=" * 70)
    print("📂 CARICAMENTO DATI")
    print("=" * 70)
    
    # Caricamento dataset
    df = pd.read_csv(filepath)
    
    print(f"\n✅ Dataset caricato: {len(df)} record")
    print(f"📊 Colonne: {list(df.columns)}")
    
    # Statistiche descrittive
    print("\n📈 Distribuzione delle variabili:")
    print("-" * 50)
    
    for col in df.columns:
        print(f"\n🔹 {col}:")
        counts = df[col].value_counts()
        for val, count in counts.items():
            pct = count / len(df) * 100
            print(f"   {val}: {count} ({pct:.1f}%)")
    
    return df


def build_bayesian_network():
    """Costruisce la struttura del DAG usando DiscreteBayesianNetwork"""
    
    print("\n" + "=" * 70)
    print("🔗 COSTRUZIONE STRUTTURA DAG")
    print("=" * 70)
    
    # Definizione archi del DAG
    edges = [
        ('Weather_conditions', 'Road_traffic_density'),  # Meteo → Traffico
        ('Weather_conditions', 'Delivery_Status'),       # Meteo → Ritardo
        ('Road_traffic_density', 'Delivery_Status'),     # Traffico → Ritardo
        ('Age_Category', 'Delivery_Status')              # Età → Ritardo
    ]
    
    # Creazione modello con DiscreteBayesianNetwork
    model = DiscreteBayesianNetwork(edges)
    
    print("\n📐 Struttura del modello:")
    print("-" * 50)
    print("""
                    ┌─────────────────────┐
                    │ Weather_conditions  │
                    └─────────┬───────────┘
                              │
              ┌───────────────┼───────────────┐
              │               │               │
              ▼               │               │
    ┌─────────────────────┐   │               │
    │Road_traffic_density │   │               │
    └─────────┬───────────┘   │               │
              │               │               │
              │               ▼               │
              │    ┌─────────────────────┐    │
              └───►│  Delivery_Status    │◄───┘
                   └─────────────────────┘
                             ▲
                             │
                   ┌─────────────────────┐
                   │    Age_Category     │
                   └─────────────────────┘
    """)
    
    print("🔗 Archi definiti:")
    for parent, child in edges:
        print(f"   {parent} → {child}")
    
    return model


# ═══════════════════════════════════════════════════════════════════════════
# SEZIONE 2: TASK 3 - APPRENDIMENTO PARAMETRI (MLE)
# ═══════════════════════════════════════════════════════════════════════════

def train_model_mle(model, data):
    """
    Task 3: Addestra il modello usando Maximum Likelihood Estimation
    """
    
    print("\n" + "=" * 70)
    print("🎓 TASK 3: APPRENDIMENTO PARAMETRI (MLE)")
    print("=" * 70)
    
    # Training con MLE
    print("\n⏳ Training in corso con MaximumLikelihoodEstimator...")
    
    model.fit(data, estimator=MaximumLikelihoodEstimator)
    
    print("✅ Training completato!")
    
    return model


def display_cpt_tables(model, data):
    """Visualizza le CPT (Conditional Probability Tables)"""
    
    print("\n" + "-" * 70)
    print("📊 CONDITIONAL PROBABILITY TABLES (CPT)")
    print("-" * 70)
    
    # CPT per Road_traffic_density
    print("\n" + "=" * 60)
    print("🚗 CPT: Road_traffic_density | Weather_conditions")
    print("=" * 60)
    print("(Mostra come il meteo influenza la densità del traffico)")
    print()
    
    cpd_traffic = model.get_cpds('Road_traffic_density')
    
    # Creiamo una tabella più leggibile
    weather_states = cpd_traffic.state_names['Weather_conditions']
    traffic_states = cpd_traffic.state_names['Road_traffic_density']
    
    # Header
    print(f"{'Traffico':<15} | ", end="")
    for w in weather_states:
        print(f"{w:<12} | ", end="")
    print()
    print("-" * (16 + 15 * len(weather_states)))
    
    # Valori
    values = cpd_traffic.get_values()
    for i, traffic in enumerate(traffic_states):
        print(f"{traffic:<15} | ", end="")
        for j in range(len(weather_states)):
            print(f"{values[i][j]*100:>10.1f}% | ", end="")
        print()
    
    print("\n💡 Interpretazione:")
    print("   - La tabella mostra P(Traffico | Meteo)")
    print("   - Ogni colonna somma a 100%")
    
    # CPT per Delivery_Status
    print("\n" + "=" * 60)
    print("📦 CPT: Delivery_Status | Weather, Traffic, Age")
    print("=" * 60)
    print("(Mostra come i fattori influenzano il ritardo)")
    print()
    
    cpd_delivery = model.get_cpds('Delivery_Status')
    
    # Per questa CPT complessa, mostriamo un riassunto
    print("📋 Tabella completa (formato pgmpy):")
    print("-" * 60)
    print(cpd_delivery)
    
    # Mostriamo alcuni casi specifici interessanti
    print("\n📊 Casi rappresentativi (probabilità di RITARDO):")
    print("-" * 75)
    
    delivery_states = cpd_delivery.state_names['Delivery_Status']
    weather_states_del = cpd_delivery.state_names['Weather_conditions']
    traffic_states_del = cpd_delivery.state_names['Road_traffic_density']
    age_states = cpd_delivery.state_names['Age_Category']
    
    # Definiamo casi interessanti basati sui dati reali
    interesting_cases = []
    
    # Trova combinazioni esistenti nei dati
    for weather in weather_states_del[:3]:  # Prime 3 condizioni meteo
        for traffic in traffic_states_del[:2]:  # Prime 2 condizioni traffico
            for age in age_states[:2]:  # Prime 2 categorie età
                interesting_cases.append((weather, traffic, age))
    
    print(f"\n{'Meteo':<12} | {'Traffico':<8} | {'Età':<10} | {'P(Ritardo)':<12}")
    print("-" * 55)
    
    for weather, traffic, age in interesting_cases[:6]:  # Mostra max 6 casi
        try:
            prob = cpd_delivery.get_value(
                Delivery_Status='Ritardo',
                Weather_conditions=weather,
                Road_traffic_density=traffic,
                Age_Category=age
            )
            print(f"{weather:<12} | {traffic:<8} | {age:<10} | {prob*100:>10.1f}%")
        except:
            pass  # Salta combinazioni non valide
    
    return cpd_traffic, cpd_delivery


def validate_model(model):
    """Verifica la validità del modello"""
    
    print("\n" + "-" * 70)
    print("✔️  VALIDAZIONE MODELLO")
    print("-" * 70)
    
    try:
        is_valid = model.check_model()
        if is_valid:
            print("✅ Il modello è VALIDO!")
            print("   - Tutte le CPT sono correttamente definite")
            print("   - Le probabilità sommano a 1 per ogni configurazione")
            print("   - Il DAG è aciclico")
        return is_valid
    except Exception as e:
        print(f"❌ Errore nella validazione: {e}")
        return False


# ═══════════════════════════════════════════════════════════════════════════
# SEZIONE 3: TASK 4 - MOTORE DI INFERENZA
# ═══════════════════════════════════════════════════════════════════════════

def setup_inference_engine(model):
    """Inizializza il motore di inferenza"""
    
    print("\n" + "=" * 70)
    print("🔮 TASK 4: MOTORE DI INFERENZA")
    print("=" * 70)
    
    print("\n⚙️  Inizializzazione Variable Elimination...")
    inference = VariableElimination(model)
    print("✅ Motore di inferenza pronto!")
    
    return inference


def execute_query(inference, query_var, evidence, scenario_name, scenario_desc):
    """
    Esegue una query probabilistica e formatta i risultati
    """
    
    print("\n" + "─" * 70)
    print(f"🔍 {scenario_name}")
    print("─" * 70)
    print(f"📝 Domanda: {scenario_desc}")
    
    print(f"\n📋 Evidenze fornite:")
    for var, val in evidence.items():
        print(f"   • {var} = '{val}'")
    
    print(f"\n🎯 Query: P({query_var} | evidenze)")
    
    # Esecuzione query
    result = inference.query(variables=[query_var], evidence=evidence)
    
    # Estrazione risultati
    states = result.state_names[query_var]
    values = result.values
    
    print(f"\n📊 RISULTATI:")
    print("-" * 40)
    
    # Formattazione risultati con barra visuale
    max_prob = max(values)
    
    for state, prob in zip(states, values):
        bar_length = int(prob * 30)
        bar = "█" * bar_length + "░" * (30 - bar_length)
        
        # Indicatore per il più probabile
        if prob == max_prob:
            indicator = "◄── PIÙ PROBABILE"
        else:
            indicator = ""
        
        print(f"   {state:<15}: {prob*100:>6.2f}% |{bar}| {indicator}")
    
    return result


def run_inference_scenarios(inference, data):
    """Esegue tutti gli scenari di test"""
    
    print("\n" + "=" * 70)
    print("🎬 ESECUZIONE SCENARI DI INFERENZA")
    print("=" * 70)
    
    # Ottieni i valori unici dal dataset per usare evidenze valide
    weather_values = data['Weather_conditions'].unique().tolist()
    traffic_values = data['Road_traffic_density'].unique().tolist()
    age_values = data['Age_Category'].unique().tolist()
    
    print(f"\n📋 Stati disponibili nel dataset:")
    print(f"   • Weather: {weather_values}")
    print(f"   • Traffic: {traffic_values}")
    print(f"   • Age: {age_values}")
    
    # ═══════════════════════════════════════════════════════════════════
    # SCENARIO A: Analisi Rischio - Predizione (Caso Pessimo)
    # ═══════════════════════════════════════════════════════════════════
    
    scenario_a = execute_query(
        inference=inference,
        query_var='Delivery_Status',
        evidence={
            'Weather_conditions': 'Stormy',
            'Age_Category': 'Giovane', 
            'Road_traffic_density': 'Jam'
        },
        scenario_name="SCENARIO A: Analisi Rischio - PREDIZIONE (Caso Estremo)",
        scenario_desc="Dato che c'è TEMPESTA, l'autista è GIOVANE e c'è INGORGO,\n           qual è la probabilità di Ritardo?"
    )
    
    # Interpretazione
    print("\n💡 Interpretazione Scenario A:")
    prob_ritardo = scenario_a.get_value(Delivery_Status='Ritardo')
    prob_puntuale = scenario_a.get_value(Delivery_Status='Puntuale')
    
    if prob_ritardo > 0.5:
        print(f"   🔴 ALTO RISCHIO! Con {prob_ritardo*100:.1f}% di probabilità di ritardo,")
        print("   questa combinazione è sfavorevole per la puntualità.")
    else:
        print(f"   🟢 Rischio contenuto: {prob_ritardo*100:.1f}% di ritardo")
        print(f"   Il modello stima {prob_puntuale*100:.1f}% di puntualità")
    
    # ═══════════════════════════════════════════════════════════════════
    # SCENARIO B: Situazione Ideale (Caso Ottimo)
    # ═══════════════════════════════════════════════════════════════════
    
    scenario_b = execute_query(
        inference=inference,
        query_var='Delivery_Status',
        evidence={
            'Weather_conditions': 'Sunny',
            'Age_Category': 'Senior',
            'Road_traffic_density': 'Low'
        },
        scenario_name="SCENARIO B: Analisi Rischio - SITUAZIONE IDEALE",
        scenario_desc="Dato che c'è SOLE, l'autista è SENIOR e il traffico è BASSO,\n           qual è la probabilità di Ritardo?"
    )
    
    # Interpretazione
    print("\n💡 Interpretazione Scenario B:")
    prob_ritardo_b = scenario_b.get_value(Delivery_Status='Ritardo')
    prob_puntuale_b = scenario_b.get_value(Delivery_Status='Puntuale')
    
    if prob_puntuale_b > 0.7:
        print(f"   🟢 CONDIZIONI OTTIMALI! Con {prob_puntuale_b*100:.1f}% di puntualità,")
        print("   questa è la combinazione ideale per le consegne.")
    else:
        print(f"   🟡 Puntualità stimata: {prob_puntuale_b*100:.1f}%")
    
    # Confronto A vs B
    prob_ritardo_a = scenario_a.get_value(Delivery_Status='Ritardo')
    print("\n📈 CONFRONTO Scenario A vs B:")
    print(f"   Rischio Ritardo: {prob_ritardo_a*100:.1f}% (pessimo) vs {prob_ritardo_b*100:.1f}% (ottimo)")
    print(f"   Differenza: {abs(prob_ritardo_a - prob_ritardo_b)*100:.1f} punti percentuali!")
    
    # ═══════════════════════════════════════════════════════════════════
    # SCENARIO C: Diagnosi - Ragionamento Inverso (Sherlock Holmes)
    # ═══════════════════════════════════════════════════════════════════
    
    scenario_c = execute_query(
        inference=inference,
        query_var='Road_traffic_density',
        evidence={
            'Delivery_Status': 'Ritardo',
            'Age_Category': 'Adulto'
        },
        scenario_name="SCENARIO C: DIAGNOSI - Ragionamento Inverso (Sherlock Holmes)",
        scenario_desc="Il pacco è arrivato in RITARDO e l'autista era ESPERTO (Adulto).\n           Qual era la probabile densità del traffico?"
    )
    
    # Interpretazione
    print("\n💡 Interpretazione Scenario C (Ragionamento Diagnostico):")
    
    # Calcola probabilità per ogni tipo di traffico
    traffic_probs = {}
    for traffic in traffic_values:
        traffic_probs[traffic] = scenario_c.get_value(Road_traffic_density=traffic)
    
    # Trova il più probabile
    most_likely = max(traffic_probs, key=traffic_probs.get)
    print(f"   🔍 Traffico più probabile dato il ritardo: {most_likely}")
    print(f"      con probabilità {traffic_probs[most_likely]*100:.1f}%")
    
    # Se Jam e High esistono, calcola probabilità combinata
    prob_traffic_alto = 0
    if 'Jam' in traffic_probs:
        prob_traffic_alto += traffic_probs['Jam']
    if 'High' in traffic_probs:
        prob_traffic_alto += traffic_probs['High']
    
    if prob_traffic_alto > 0:
        print(f"   📊 Probabilità traffico intenso (Jam + High): {prob_traffic_alto*100:.1f}%")
    
    return scenario_a, scenario_b, scenario_c


# ═══════════════════════════════════════════════════════════════════════════
# SEZIONE 4: ANALISI AGGIUNTIVE
# ═══════════════════════════════════════════════════════════════════════════

def additional_analysis(inference, data):
    """Esegue analisi aggiuntive interessanti"""
    
    print("\n" + "=" * 70)
    print("📊 ANALISI AGGIUNTIVE")
    print("=" * 70)
    
    # Ottieni valori unici
    weather_conditions = data['Weather_conditions'].unique().tolist()
    age_categories = data['Age_Category'].unique().tolist()
    
    # Analisi: Effetto del solo meteo
    print("\n🌤️  Effetto del METEO sul rischio di ritardo:")
    print("-" * 50)
    
    weather_emojis = {
        'Sunny': '☀️', 'Cloudy': '☁️', 'Fog': '🌫️', 
        'Stormy': '⛈️', 'Windy': '💨', 'Sandstorms': '🏜️'
    }
    
    print(f"\n{'Meteo':<15} | {'P(Ritardo)':<12} | {'P(Puntuale)':<12} | Rischio")
    print("-" * 60)
    
    for weather in sorted(weather_conditions):
        try:
            result = inference.query(
                variables=['Delivery_Status'],
                evidence={'Weather_conditions': weather}
            )
            prob_delay = result.get_value(Delivery_Status='Ritardo')
            prob_punct = result.get_value(Delivery_Status='Puntuale')
            
            if prob_delay > 0.5:
                risk = "🔴 ALTO"
            elif prob_delay > 0.3:
                risk = "🟠 MEDIO"
            else:
                risk = "🟢 BASSO"
            
            emoji = weather_emojis.get(weather, '🌡️')
            print(f"{emoji} {weather:<12} | {prob_delay*100:>10.1f}% | {prob_punct*100:>10.1f}% | {risk}")
        except Exception as e:
            print(f"   ⚠️ Errore per {weather}: {e}")
    
    # Analisi: Effetto dell'esperienza
    print("\n\n👤 Effetto dell'ESPERIENZA dell'autista:")
    print("-" * 50)
    
    age_emojis = {'Giovane': '🧑', 'Adulto': '👨', 'Senior': '👴'}
    
    print(f"\n{'Categoria':<12} | {'P(Ritardo)':<12} | {'P(Puntuale)':<12} | Affidabilità")
    print("-" * 65)
    
    for age in sorted(age_categories):
        try:
            result = inference.query(
                variables=['Delivery_Status'],
                evidence={'Age_Category': age}
            )
            prob_delay = result.get_value(Delivery_Status='Ritardo')
            prob_punct = result.get_value(Delivery_Status='Puntuale')
            
            if prob_punct > 0.7:
                reliability = "⭐⭐⭐ ALTA"
            elif prob_punct > 0.5:
                reliability = "⭐⭐ MEDIA"
            else:
                reliability = "⭐ BASSA"
            
            emoji = age_emojis.get(age, '👤')
            print(f"{emoji} {age:<10} | {prob_delay*100:>10.1f}% | {prob_punct*100:>10.1f}% | {reliability}")
        except Exception as e:
            print(f"   ⚠️ Errore per {age}: {e}")
    
    # Analisi: Query diagnostica sul meteo
    print("\n\n🔮 DIAGNOSI: Se c'è ritardo, qual era probabilmente il meteo?")
    print("-" * 50)
    
    try:
        result = inference.query(
            variables=['Weather_conditions'],
            evidence={'Delivery_Status': 'Ritardo'}
        )
        
        print(f"\n{'Meteo':<15} | P(Meteo | Ritardo)")
        print("-" * 40)
        
        for weather in sorted(weather_conditions):
            prob = result.get_value(Weather_conditions=weather)
            bar = "█" * int(prob * 25)
            emoji = weather_emojis.get(weather, '🌡️')
            print(f"{emoji} {weather:<12} | {prob*100:>6.1f}% |{bar}")
    except Exception as e:
        print(f"   ⚠️ Errore nella query diagnostica: {e}")


def print_summary(data):
    """Stampa un riepilogo finale"""
    
    print("\n" + "=" * 70)
    print("📋 RIEPILOGO ESECUZIONE")
    print("=" * 70)
    
    # Calcola statistiche dal dataset
    total_records = len(data)
    puntuale_count = (data['Delivery_Status'] == 'Puntuale').sum()
    ritardo_count = (data['Delivery_Status'] == 'Ritardo').sum()
    
    print(f"""
    ✅ Task 3 Completato:
       • Modello addestrato con Maximum Likelihood Estimation
       • CPT generate per tutti i nodi
       • Modello validato con successo
       • Dataset utilizzato: {total_records} record
         - Puntuali: {puntuale_count} ({puntuale_count/total_records*100:.1f}%)
         - Ritardi: {ritardo_count} ({ritardo_count/total_records*100:.1f}%)
    
    ✅ Task 4 Completato:
       • Motore di inferenza Variable Elimination configurato
       • Scenario A (Predizione Pessimistica): Analizzato
       • Scenario B (Predizione Ottimistica): Analizzato  
       • Scenario C (Diagnosi Inversa): Analizzato
    
    🎯 Insight Principali:
       • La rete bayesiana permette ragionamento probabilistico bidirezionale
       • Predizione: dalle cause (meteo, età) agli effetti (ritardo)
       • Diagnosi: dagli effetti (ritardo) alle probabili cause
    """)


# ═══════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════════════

def main():
    """Funzione principale che orchestra l'esecuzione"""
    
    print("╔" + "═" * 68 + "╗")
    print("║" + " " * 15 + "RETE BAYESIANA - PARTE 2" + " " * 28 + "║")
    print("║" + " " * 10 + "Training e Inferenza Probabilistica" + " " * 22 + "║")
    print("╚" + "═" * 68 + "╝")
    
    # Step 1: Caricamento dati
    data = load_and_prepare_data('bayesian_training_data.csv')
    
    # Step 2: Costruzione struttura modello
    model = build_bayesian_network()
    
    # Step 3: Training con MLE
    model = train_model_mle(model, data)
    
    # Visualizzazione CPT
    display_cpt_tables(model, data)
    
    # Validazione
    validate_model(model)
    
    # Step 4: Inferenza
    inference = setup_inference_engine(model)
    
    # Esecuzione scenari
    run_inference_scenarios(inference, data)
    
    # Analisi aggiuntive
    additional_analysis(inference, data)
    
    # Riepilogo
    print_summary(data)
    
    return model, inference


# ═══════════════════════════════════════════════════════════════════════════
# ESECUZIONE
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    model, inference = main()

╔════════════════════════════════════════════════════════════════════╗
║               RETE BAYESIANA - PARTE 2                            ║
║          Training e Inferenza Probabilistica                      ║
╚════════════════════════════════════════════════════════════════════╝
📂 CARICAMENTO DATI

✅ Dataset caricato: 39997 record
📊 Colonne: ['Weather_conditions', 'Road_traffic_density', 'Age_Category', 'Delivery_Status']

📈 Distribuzione delle variabili:
--------------------------------------------------

🔹 Weather_conditions:
   Fog: 6784 (17.0%)
   Stormy: 6757 (16.9%)
   Cloudy: 6676 (16.7%)
   Sandstorms: 6652 (16.6%)
   Windy: 6624 (16.6%)
   Sunny: 6504 (16.3%)

🔹 Road_traffic_density:
   Low: 13699 (34.3%)
   Jam: 12610 (31.5%)
   Medium: 9755 (24.4%)
   High: 3933 (9.8%)

🔹 Age_Category:
   Senior: 14187 (35.5%)
   Giovane: 13869 (34.7%)
   Adulto: 11941 (29.9%)

🔹 Delivery_Status:
   Ritardo: 30125 (75.3%)
   Puntuale: 9872 (24.7%)

🔗 COSTRUZIONE STRUTTURA DAG


ImportError: BayesianNetwork has been deprecated. Please use DiscreteBayesianNetwork instead.